# Wug paper figures & stimuliEverything in this notebook uses **existing** results / data in the repo. Run it with the`wug-test-interp` conda env kernel, from the repo root.Contents:1. PDF figure: column of singular wug images, column of plural wug images2. LaTeX table of image-condition stimuli3. LaTeX table of text (syntax) condition stimuli4. LaTeX table of the nouns used to initialize the mean-embedding cluster5. The full dev set (evaluation minimal pairs), as a table and a LaTeX `longtable`6. Averaged training loss curves (text vs. vision) for the 2B and 4B models7. Free-form generation from Qwen3-VL-2B with the learned syntax `[wug]`/`[wugs]` embeddings8. The chat template, rendered on a real training example9. LR sweep: agreement accuracy with CIs, per model x condition10. Chosen LR (0.001): 50-seed accuracy with CIs, per model x condition11. Interp seed replicates: completeness check + AvgOdds with CIsSections 9-11 are pure pandas/matplotlib over existing results and do **not** need the modelloaded in section 7 - they can be run straight after section 6.

In [ ]:
import osimport sysimport globimport textwrapimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom PIL import ImageREPO = os.path.abspath(".")assert os.path.isdir(os.path.join(REPO, "core")), f"Run from repo root, got {REPO}"if REPO not in sys.path:    sys.path.insert(0, REPO)CACHE_DIR = "/mnt/dv/wid/projects3/Rogers-muri-human-ai/zstuddiford"# Data / results paths (all pre-existing in the repo)IMAGE_DIR      = "data/embeddings/train/im/creature_1"IMAGE_TRAIN_CSV = "data/embeddings/train/text/image_train_1.csv"SYNTAX_TRAIN_CSV = "data/embeddings/train/text/syntax_train_1.csv"NOUN_INIT_TXT  = "data/embeddings/init/noun_init.txt"DEV_EVAL_CSV   = "data/embeddings/dev/dev_eval.csv"CI_ROOT        = "results/train/CI_seed_runs"FIG_DIR = "figures"os.makedirs(FIG_DIR, exist_ok=True)

## 1. Figure: singular vs. plural wug imagesTwo columns (singular on the left, plural on the right), one row per image index.Saved as a vector PDF.

In [ ]:
singular_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "singular*.png")))plural_paths   = sorted(glob.glob(os.path.join(IMAGE_DIR, "plural*.png")))print(f"{len(singular_paths)} singular, {len(plural_paths)} plural images from {IMAGE_DIR}")n_rows = max(len(singular_paths), len(plural_paths))fig, axes = plt.subplots(n_rows, 2, figsize=(4.0, 2.0 * n_rows))axes = np.atleast_2d(axes)for col, (paths, title) in enumerate([(singular_paths, "singular"), (plural_paths, "plural")]):    for row in range(n_rows):        ax = axes[row, col]        ax.set_xticks([]); ax.set_yticks([])        for spine in ax.spines.values():            spine.set_visible(False)        if row < len(paths):            ax.imshow(Image.open(paths[row]).convert("RGB"))        else:            ax.axis("off")        if row == 0:            ax.set_title(f"[wug] ({title})" if col == 0 else f"[wugs] ({title})", fontsize=11)fig.subplots_adjust(wspace=0.02, hspace=0.02)out_pdf = os.path.join(FIG_DIR, "wug_stimuli_images.pdf")fig.savefig(out_pdf, bbox_inches="tight", dpi=300)print("wrote", out_pdf)plt.show()

## 2–3. LaTeX tables of the training stimuli`escape_latex` handles the `[wug]` brackets and any stray specials; the two conditions shareone formatter so the tables come out identical in structure.

In [ ]:
def escape_latex(s):    """Escape LaTeX specials and typeset [wug]/[wugs] literally."""    repl = {        "\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",        "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",        "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",    }    out = "".join(repl.get(ch, ch) for ch in str(s))    return out.replace("[", "{[}").replace("]", "{]}")def stimuli_table(csv_path, caption, label):    """Side-by-side singular/plural stimuli table from a training CSV."""    df = pd.read_csv(csv_path)    df["type"] = df["type"].astype(str).str.strip().str.lower()    df["sentence"] = df["sentence"].astype(str).str.strip()    sing = df.loc[df["type"] == "singular", "sentence"].tolist()    plur = df.loc[df["type"] == "plural", "sentence"].tolist()    n = max(len(sing), len(plur))    sing += [""] * (n - len(sing))    plur += [""] * (n - len(plur))    lines = [        r"\begin{table}[t]",        r"\centering",        r"\small",        r"\begin{tabular}{@{}p{0.44\linewidth}p{0.44\linewidth}@{}}",        r"\toprule",        r"Singular ([wug]) & Plural ([wugs]) \\".replace("[", "{[}").replace("]", "{]}"),        r"\midrule",    ]    for s, p in zip(sing, plur):        lines.append(f"{escape_latex(s)} & {escape_latex(p)} \\\\")    lines += [        r"\bottomrule",        r"\end{tabular}",        rf"\caption{{{caption}}}",        rf"\label{{{label}}}",        r"\end{table}",    ]    return "\n".join(lines)

### 2. Image-condition stimuli

In [ ]:
print(stimuli_table(    IMAGE_TRAIN_CSV,    caption="Training stimuli for the image condition. Each sentence is paired with a "            "singular or plural creature image.",    label="tab:stimuli-image",))

### 3. Text (syntax) condition stimuli

In [ ]:
print(stimuli_table(    SYNTAX_TRAIN_CSV,    caption="Training stimuli for the text (syntax) condition. No image is shown; the novel "            "token must be learned from syntactic context alone.",    label="tab:stimuli-syntax",))

## 4. Nouns used to initialize the embedding cluster`[wug]`/`[wugs]` are initialized near the mean of these nouns' embeddings, with noise scaledto the mean singular–plural distance (see `core/train/embed_train.py --embed_init`).

In [ ]:
with open(NOUN_INIT_TXT) as f:    init_words = [w.strip() for w in f if w.strip()]print(f"{len(init_words)} initialization words\n")print(", ".join(init_words))print()N_COLS = 6rows = [init_words[i:i + N_COLS] for i in range(0, len(init_words), N_COLS)]lines = [    r"\begin{table}[t]",    r"\centering",    r"\small",    r"\begin{tabular}{@{}" + "l" * N_COLS + r"@{}}",    r"\toprule",]for row in rows:    row = row + [""] * (N_COLS - len(row))    lines.append(" & ".join(escape_latex(w) for w in row) + r" \\")lines += [    r"\bottomrule",    r"\end{tabular}",    r"\caption{The " + str(len(init_words)) + r" nouns whose mean embedding defines the "    r"initialization cluster for the novel tokens {[}wug{]} and {[}wugs{]}.}",    r"\label{tab:noun-init}",    r"\end{table}",]print("\n".join(lines))

## 5. The dev set (evaluation minimal pairs)`data/embeddings/dev/dev_eval.csv` is the minimal-pair set the agreement eval scores: each rowis a grammatical `good` sentence and its ungrammatical `bad` counterpart, and accuracy is thefraction of pairs where the model gives `good` the higher sequence score. Items are labelledsingular vs. plural by which novel token appears in the `good` sentence, matching the`singular_eval_idx` / `plural_eval_idx` split in `core/train/embed_train.py`.Printed twice: a readable pandas table, then a LaTeX `longtable` (a few hundred rows, so it hasto break across pages -- needs `\usepackage{longtable,booktabs}` in the preamble).

In [ ]:
dev = pd.read_csv(DEV_EVAL_CSV)dev["good"] = dev["good"].astype(str).str.strip()dev["bad"]  = dev["bad"].astype(str).str.strip()dev["type"] = np.where(dev["good"].str.contains(r"\[wug\]", regex=True), "singular", "plural")print(f"{len(dev)} minimal pairs from {DEV_EVAL_CSV}")print(dev["type"].value_counts().to_string())print()with pd.option_context("display.max_rows", None, "display.max_colwidth", 80,                       "display.width", 220):    print(dev[["type", "good", "bad"]].to_string(index=True))

In [ ]:
CAPTION = (r"The full dev set: " + str(len(dev)) + r" grammatical/ungrammatical minimal pairs "           r"used to evaluate the learned {[}wug{]}/{[}wugs{]} embeddings.")HEADER = r"\# & Type & Grammatical (good) & Ungrammatical (bad) \\"lines = [    r"\begin{longtable}{@{}rlp{0.38\linewidth}p{0.38\linewidth}@{}}",    r"\caption{" + CAPTION + r"}\label{tab:dev-set}\\",    r"\toprule",    HEADER,    r"\midrule",    r"\endfirsthead",    r"\multicolumn{4}{@{}l}{\small\itshape (continued from previous page)}\\",    r"\toprule",    HEADER,    r"\midrule",    r"\endhead",    r"\midrule",    r"\multicolumn{4}{r@{}}{\small\itshape (continued on next page)}\\",    r"\endfoot",    r"\bottomrule",    r"\endlastfoot",]for i, row in enumerate(dev.itertuples(index=False), start=1):    lines.append(f"{i} & {row.type} & {escape_latex(row.good)} & {escape_latex(row.bad)} \\\\")lines.append(r"\end{longtable}")print("\n".join(lines))

## 6. Averaged loss curves (existing CI seed runs)Reads `epoch_stats.csv` from every seed under `results/train/CI_seed_runs/`. Runs early-stop atdifferent epochs, so each epoch is averaged over whatever seeds are still alive there; theshaded band is a 95% CI over seeds.**Note on `avg_ce`:** the image-condition CI runs were written by an older version of`embed_train.py` and their `epoch_stats.csv` has an *empty* `avg_ce` column (`sing_ce`/`plur_ce`are fine). Plotting `avg_ce` directly therefore silently drops the vision curves. The loaderbelow falls back to `step_stats.csv`, where per-step `ce_loss` averaged within an epochreproduces `avg_ce` exactly on runs that have both — so text and vision stay comparable.

In [ ]:
CI_RUNS = {    ("2B", "text"):   f"{CI_ROOT}/lr_ci_results_Qwen3-VL-2B-Instruct_text_lr0p001_50seeds",    ("2B", "vision"): f"{CI_ROOT}/lr_ci_results_Qwen3-VL-2B-Instruct_image_lr0p001_50seeds",    ("4B", "text"):   f"{CI_ROOT}/lr_ci_results_Qwen3-VL-4B-Instruct_text_lr0p001_50seeds",    ("4B", "vision"): f"{CI_ROOT}/lr_ci_results_Qwen3-VL-4B-Instruct_image_lr0p001_50seeds",}def load_epoch_stats(run_dir):    """Stack per-epoch stats across all seeds in a CI run dir.    Returns a frame with columns [seed, epoch, avg_ce, sing_ce, plur_ce, ...]. Where    epoch_stats.csv has no usable avg_ce (the image runs), it is rebuilt as the mean    per-step ce_loss within each epoch from step_stats.csv.    """    paths = sorted(glob.glob(os.path.join(run_dir, "seed_*", "*", "epoch_stats.csv")))    frames, n_rebuilt = [], 0    for p in paths:        seed = os.path.basename(os.path.dirname(os.path.dirname(p))).replace("seed_", "")        df = pd.read_csv(p)        df["seed"] = seed        if "avg_ce" not in df.columns or df["avg_ce"].isna().all():            step_path = os.path.join(os.path.dirname(p), "step_stats.csv")            if os.path.exists(step_path):                rebuilt = pd.read_csv(step_path).groupby("epoch")["ce_loss"].mean()                df["avg_ce"] = df["epoch"].map(rebuilt)                n_rebuilt += 1            else:                print(f"  !! {p}: no avg_ce and no step_stats.csv to rebuild from")        frames.append(df)    if not frames:        print(f"  !! no epoch_stats.csv under {run_dir}")        return None    out = pd.concat(frames, ignore_index=True)    note = f", avg_ce rebuilt from step_stats for {n_rebuilt}" if n_rebuilt else ""    print(f"  {os.path.basename(run_dir)}: {len(paths)} seeds, "          f"max epoch {out['epoch'].max()}{note}")    return outstats = {}for key, d in CI_RUNS.items():    stats[key] = load_epoch_stats(d)# Sanity check: nothing should be silently all-NaN in the column we plot.for key, df in stats.items():    if df is not None:        frac = df["avg_ce"].notna().mean()        print(f"{key}: avg_ce non-null {frac:.1%}")        assert frac > 0, f"{key} has no usable avg_ce"

In [ ]:
LOSS_COL = "avg_ce"   # per-epoch mean cross-entropy on the novel-token positionsCOLORS = {"text": "#1f77b4", "vision": "#d62728"}fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), sharey=True)for ax, model in zip(axes, ["2B", "4B"]):    for cond in ["text", "vision"]:        df = stats.get((model, cond))        if df is None:            continue        df = df.dropna(subset=[LOSS_COL])        if df.empty:            print(f"!! no {LOSS_COL} data for {model}/{cond} -- nothing plotted")            continue        g = df.groupby("epoch")[LOSS_COL]        mean, sd, n = g.mean(), g.std(ddof=1), g.count()        ci = 1.96 * sd / np.sqrt(n.clip(lower=1))        ax.plot(mean.index, mean.values, color=COLORS[cond], lw=1.8,                label=f"{cond} (n={int(n.max())} seeds)")        ax.fill_between(mean.index, mean - ci, mean + ci, color=COLORS[cond], alpha=0.20, lw=0)    ax.set_title(f"Qwen3-VL-{model}")    ax.set_xlabel("epoch")    ax.legend(frameon=False, fontsize=9)    ax.spines[["top", "right"]].set_visible(False)axes[0].set_ylabel("mean CE on novel token")fig.tight_layout()out_pdf = os.path.join(FIG_DIR, "loss_curves_text_vs_vision.pdf")fig.savefig(out_pdf, bbox_inches="tight")print("wrote", out_pdf)plt.show()

## 7. Free-form generation with the learned syntax embeddings (Qwen3-VL-2B)Loading mirrors `core/interp/intervention.py::_inject_embeddings`:1. build the `VLMScorer` with `cache_dir=CACHE_DIR`,2. add the `" [wug]"` / `" [wugs]"` tokens and resize the embedding matrix,3. write the saved `wug_embedding` / `wugs_embedding` rows into `embed_tokens`.Embeddings and `lm_head` are tied, so the injected rows are both readable and generatable.

In [ ]:
import torchfrom minicons import scorerfrom utils.chat_templates import train_chat_template_noimageMODEL_PATH      = "Qwen/Qwen3-VL-2B-Instruct"EMBEDDINGS_PATH = "embeddings/Qwen3-VL-2B/syntax/qwen3_vl_2b_syntax.pt"ADDED_TOKENS    = [" [wug]", " [wugs]"]device = "cuda" if torch.cuda.is_available() else "cpu"lm = scorer.VLMScorer(MODEL_PATH, device=device, torch_dtype=torch.bfloat16,                      cache_dir=CACHE_DIR)tok = lm.tokenizer.tokenizer          # inner tokenizer; lm.tokenizer is the processormodel = lm.modelto_add = [t for t in ADDED_TOKENS if t not in tok.get_vocab()]if to_add:    tok.add_tokens(to_add)    old_len = model.resize_token_embeddings().weight.shape[0]    model.resize_token_embeddings(old_len + len(to_add))emb = model.model.language_model.embed_tokenswug_id, wugs_id = [tok(t, add_special_tokens=False).input_ids[0] for t in ADDED_TOKENS]rec = torch.load(EMBEDDINGS_PATH, map_location="cpu")with torch.no_grad():    emb.weight.data[wug_id]  = rec["wug_embedding"].to(emb.weight.device, dtype=emb.weight.dtype)    emb.weight.data[wugs_id] = rec["wugs_embedding"].to(emb.weight.device, dtype=emb.weight.dtype)print(f"injected {EMBEDDINGS_PATH}")print(f"  saved model  : {rec.get('model_name')}   epoch: {rec.get('saved_epoch')}")print(f"  ids in file  : wug={rec['wug_id']} wugs={rec['wugs_id']}")print(f"  ids here     : wug={wug_id} wugs={wugs_id}")assert (rec["wug_id"], rec["wugs_id"]) == (wug_id, wugs_id), \    "Token id mismatch between checkpoint and current tokenizer"print(f"  tied lm_head : {emb.weight.data_ptr() == model.lm_head.weight.data_ptr()}")

In [ ]:
PROMPTS = [    "One [wug] was playing and another came to join it. Now there are",    "I saw a single [wug] yesterday. Today I saw three",    "There is one [wug] on the left and two",    "Every [wug] in the field looked up. All of the",    "The [wugs] were resting, but only one",]@torch.no_grad()def generate(sentence, max_new_tokens=10, do_sample=False):    """Greedy continuation, using the same user turn the model was trained with."""    context = [        {"role": "user", "content": [{"type": "text", "text": "Complete the sentence."}]},        {"role": "assistant", "content": [{"type": "text", "text": sentence}]},    ]    prompt = lm.tokenizer.apply_chat_template(context, continue_final_message=True)    enc = lm.tokenizer(text=[prompt], return_tensors="pt", padding=True)    enc = {k: v.to(model.device) for k, v in enc.items()}    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,                         pad_token_id=tok.pad_token_id or tok.eos_token_id)    new_ids = out[0, enc["input_ids"].shape[1]:]    return tok.decode(new_ids, skip_special_tokens=True)for sent in PROMPTS:    cont = generate(sent)    print(f"PROMPT     : {sent}")    print(f"CONTINUATION: {cont!r}")    print(f"FULL       : {sent}{cont}")    print("-" * 78)

## 8. The chat template, rendered on a training example`train_chat_template_noimage` is what the syntax condition trains on; the tokenization showsexactly where the `[wug]` token lands.

In [ ]:
example = pd.read_csv(SYNTAX_TRAIN_CSV).iloc[0]["sentence"].strip()rendered = train_chat_template_noimage(lm, example)print("Training sentence:")print(f"  {example!r}\n")print("Rendered chat template (repr, so newlines/specials are visible):")print(f"  {rendered!r}\n")print("Rendered chat template (raw):")print("-" * 78)print(rendered)print("-" * 78)ids = lm.tokenizer(text=[rendered], return_tensors="pt")["input_ids"][0]print(f"\n{len(ids)} tokens; [wug] id = {wug_id}, [wugs] id = {wugs_id}")for i, t in enumerate(ids.tolist()):    mark = "  <-- supervised novel token" if t in (wug_id, wugs_id) else ""    print(f"  {i:3d}  {t:7d}  {tok.decode([t])!r}{mark}")

---# 9. Learning-rate sweep: accuracy with CIs`results/train/LR_sweeps/` holds 14 learning rates x 5 seeds for each of the fourmodel x condition cells. Each seed's `run_summary.csv` gives `final_overall` /`final_sing` / `final_plur` (singular--plural agreement accuracy at the selected epoch).Points are the mean over seeds, error bars a 95% normal CI (`1.96 * sem`); with n=5 these arewide by construction, so read them as spread, not as significance tests.

In [ ]:
LR_ROOT = "results/train/LR_sweeps"SWEEPS = {    ("2B", "text"):   f"{LR_ROOT}/lr_sweep_results_Qwen3-VL-2B-Instruct_text_5seeds",    ("2B", "vision"): f"{LR_ROOT}/lr_sweep_results_Qwen3-VL-2B-Instruct_image_5seeds",    ("4B", "text"):   f"{LR_ROOT}/lr_sweep_results_Qwen3-VL-4B-Instruct_text_5seeds",    ("4B", "vision"): f"{LR_ROOT}/lr_sweep_results_Qwen3-VL-4B-Instruct_image_5seeds",}CHOSEN_LR = 0.001   # the LR carried forward to the 50-seed CI runsdef load_run_summaries(pattern):    """Concatenate every run_summary.csv matching a glob pattern."""    paths = sorted(glob.glob(pattern))    if not paths:        print(f"  !! nothing matched {pattern}")        return None    return pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)sweep_rows = []for (model, cond), d in SWEEPS.items():    df = load_run_summaries(os.path.join(d, "lr_*", "seed_*", "*", "run_summary.csv"))    if df is None:        continue    df["model"], df["cond"] = model, cond    sweep_rows.append(df)    print(f"  {model}/{cond}: {len(df)} runs, {df['lr'].nunique()} LRs, "          f"{df.groupby('lr').size().min()}-{df.groupby('lr').size().max()} seeds per LR")sweep = pd.concat(sweep_rows, ignore_index=True)print(f"\ntotal sweep runs: {len(sweep)}")

In [ ]:
def mean_ci(s, z=1.96):    """Mean and half-width of a 95% normal CI. NaN half-width when n < 2."""    s = pd.Series(s).dropna()    n = len(s)    if n == 0:        return np.nan, np.nan    if n < 2:        return s.mean(), np.nan    return s.mean(), z * s.std(ddof=1) / np.sqrt(n)METRIC = "final_overall"      # or final_sing / final_plurCOND_COLORS = {"text": "#1f77b4", "vision": "#d62728"}fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)for ax, model in zip(axes, ["2B", "4B"]):    for cond in ["text", "vision"]:        sub = sweep[(sweep["model"] == model) & (sweep["cond"] == cond)]        if sub.empty:            continue        agg = (sub.groupby("lr")[METRIC]                  .apply(lambda s: pd.Series(mean_ci(s), index=["mean", "ci"]))                  .unstack()                  .sort_index())        ax.errorbar(agg.index, agg["mean"], yerr=agg["ci"], marker="o", ms=4,                    capsize=3, lw=1.5, color=COND_COLORS[cond],                    label=f"{cond} (n={int(sub.groupby('lr').size().min())}/LR)")    ax.axvline(CHOSEN_LR, color="k", ls="--", lw=1, alpha=0.6)    ax.annotate(f"chosen\nlr={CHOSEN_LR}", (CHOSEN_LR, 0.02), fontsize=8,                ha="left", va="bottom", xytext=(3, 0), textcoords="offset points")    ax.axhline(0.5, color="grey", ls=":", lw=1)    ax.set_xscale("log")    ax.set_xlabel("learning rate")    ax.set_title(f"Qwen3-VL-{model}")    ax.set_ylim(0, 1.02)    ax.grid(alpha=0.25)    ax.legend(frameon=False, fontsize=9, loc="lower left")axes[0].set_ylabel(f"{METRIC}  (agreement accuracy)")fig.suptitle("LR sweep: singular/plural accuracy, mean +/- 95% CI over seeds", fontsize=11)fig.tight_layout()out_pdf = os.path.join(FIG_DIR, "lr_sweep_accuracy_ci.pdf")fig.savefig(out_pdf, bbox_inches="tight")print("wrote", out_pdf)plt.show()

## 10. Chosen LR: 50-seed evaluation with CIsThe same `run_summary.csv` fields, now from the 50-seed CI runs at the chosen lr=0.001.Overall / singular / plural accuracy per model x condition, mean +/- 95% CI over seeds, with theper-seed values overlaid so the (often bimodal) spread is visible rather than hidden by the bar.

In [ ]:
ci_rows = []for (model, cond), d in CI_RUNS.items():    df = load_run_summaries(os.path.join(d, "seed_*", "*", "run_summary.csv"))    if df is None:        continue    df["model"], df["cond"] = model, cond    ci_rows.append(df)    print(f"  {model}/{cond}: {len(df)} seeds "          f"(lr={sorted(df['lr'].unique())}, early-stopped: {int(df['stopped_early'].sum())})")final = pd.concat(ci_rows, ignore_index=True)assert set(final["lr"].unique()) == {CHOSEN_LR}, f"unexpected LRs: {final['lr'].unique()}"summary = (final.groupby(["model", "cond"])[["final_overall", "final_sing", "final_plur"]]                .agg(["mean", "count"]).round(3))print()print(summary)

In [ ]:
METRICS = ["final_overall", "final_sing", "final_plur"]LABELS  = {"final_overall": "overall", "final_sing": "singular", "final_plur": "plural"}CELLS   = [("2B", "text"), ("2B", "vision"), ("4B", "text"), ("4B", "vision")]fig, ax = plt.subplots(figsize=(9.5, 4.2))width = 0.26x = np.arange(len(CELLS))rng = np.random.default_rng(0)for j, metric in enumerate(METRICS):    means, cis = [], []    for k, (model, cond) in enumerate(CELLS):        vals = final.loc[(final["model"] == model) & (final["cond"] == cond), metric]        m, ci = mean_ci(vals)        means.append(m); cis.append(ci)        # per-seed jitter        xs = x[k] + (j - 1) * width + rng.uniform(-0.06, 0.06, len(vals))        ax.scatter(xs, vals, s=6, color="k", alpha=0.18, zorder=3, linewidths=0)    ax.bar(x + (j - 1) * width, means, width, yerr=cis, capsize=4,           label=LABELS[metric], zorder=2, alpha=0.9)ax.axhline(0.5, color="grey", ls=":", lw=1, zorder=1)ax.set_xticks(x)ax.set_xticklabels([f"{m}\n{c}" for m, c in CELLS])ax.set_ylabel("agreement accuracy")ax.set_ylim(0, 1.05)ax.set_title(f"Final embeddings at lr={CHOSEN_LR}: mean +/- 95% CI over 50 seeds "             "(dots = individual seeds)", fontsize=11)ax.legend(frameon=False, ncol=3, fontsize=9)ax.grid(axis="y", alpha=0.25)fig.tight_layout()out_pdf = os.path.join(FIG_DIR, "final_lr_accuracy_ci.pdf")fig.savefig(out_pdf, bbox_inches="tight")print("wrote", out_pdf)plt.show()

## 11. Interp seed replicates: completeness check + AvgOdds with CIs### 11a. Do we have every run?Expected grid: 2 models x 2 conditions x 4 attractor counts x 5 seeds x 7 method files.The check below reports exactly what is present and asserts nothing is missing, so a silentlyincomplete sweep fails loudly instead of producing a thinner-than-advertised plot.

In [ ]:
SEED_ROOT = "results/interp/seed_replicate"MODELS  = ["Qwen3-VL-2B", "Qwen3-VL-4B"]STREAMS = ["syntax", "vision"]CONDS   = [f"target_verb_att{a}_opp" for a in range(4)]METHOD_FILES = ["das", "diffmean", "probe",                "ablation_k128", "ablation_nodes_k128",                "patch_k128", "patch_nodes_k128"]N_SEEDS_EXPECTED = 5rows, missing = [], []for model in MODELS:    for stream in STREAMS:        for cond in CONDS:            seed_dirs = sorted(glob.glob(f"{SEED_ROOT}/{model}/{stream}/{cond}/seed_*"))            seeds = [os.path.basename(s).replace("seed_", "") for s in seed_dirs]            present = 0            for sd in seed_dirs:                for m in METHOD_FILES:                    p = os.path.join(sd, f"{m}.csv")                    if os.path.exists(p) and os.path.getsize(p) > 0:                        present += 1                    else:                        missing.append(p)            rows.append(dict(model=model, stream=stream, cond=cond,                             n_seeds=len(seed_dirs), seeds=",".join(sorted(seeds)),                             files=present,                             files_expected=N_SEEDS_EXPECTED * len(METHOD_FILES)))cov = pd.DataFrame(rows)print(cov.to_string(index=False))print(f"\ntotal files present: {cov['files'].sum()} / {cov['files_expected'].sum()}")if missing:    print(f"\nMISSING ({len(missing)}):")    for p in missing[:40]:        print("  ", p)else:    print("\nComplete: every model x stream x condition x seed x method file is present.")assert (cov["n_seeds"] == N_SEEDS_EXPECTED).all(), "some cells do not have 5 seeds"assert not missing, f"{len(missing)} result files missing"

### 11b. Loading and the odds measure`odds` follows CausalGym (Arora, Jurafsky & Potts 2024, §3.4), matching`analysis/interp/intervention_measures.ipynb`:$$\mathrm{odds} = \big[\log p(y_b \mid b) - \log p(y_s \mid b)\big] + \big[\log p^{*}(y_s \mid b,s) - \log p^{*}(y_b \mid b,s)\big]$$Two families need different summaries, so they are plotted separately rather than pooled:- **das / diffmean / probe** sweep (layer, tok), so the per-seed scalar is **MaxAvgOdds** —  the best site, i.e. `max` over cells of the within-cell mean odds.- **patch_k128** is a single k=128 circuit with no site grid, so its per-seed scalar is the  plain **AvgOdds** over rows.`ablation_k128` uses `clean_` / `circuit_ablation_` columns rather than the`base_` / `base_intervention_` pair, so the odds formula above does not apply unmodified; it iscounted in the completeness check but left out of the odds plot.Rows with a non-empty `error` are dropped (per-cell failures written as `-1.0`), as in`analysis/interp/full_interp.ipynb`. diffmean fails on a substantial share of cells — the droprate is reported below so it is not mistaken for a real effect.

In [ ]:
SITE_METHODS = ["das", "diffmean", "probe"]CIRCUIT_METHODS = ["patch_k128"]SPLIT = "test"      # "test" = held-out wug items; "train" = natural itemsdef add_odds(df):    df = df.copy()    df["odds"] = ((df["base_logp_B"] - df["base_logp_A"])                  + (df["base_intervention_logp_A"] - df["base_intervention_logp_B"]))    return dfdef load_interp(path):    """Read one result CSV, drop failed cells, add odds. Returns (df, err_fraction)."""    df = pd.read_csv(path)    err_frac = 0.0    if "error" in df.columns:        bad = df["error"].notna() & (df["error"].astype(str).str.strip() != "")        err_frac = float(bad.mean())        df = df[~bad]    if df.empty:        return None, err_frac    return add_odds(df), err_fracrecords = []for model in MODELS:    for stream in STREAMS:        for att, cond in enumerate(CONDS):            for sd in sorted(glob.glob(f"{SEED_ROOT}/{model}/{stream}/{cond}/seed_*")):                seed = os.path.basename(sd).replace("seed_", "")                for m in SITE_METHODS + CIRCUIT_METHODS:                    df, err_frac = load_interp(os.path.join(sd, f"{m}.csv"))                    if df is None:                        continue                    d = df[df["split"] == SPLIT]                    if d.empty:                        continue                    if m in SITE_METHODS:                        val = d.groupby(["layer", "tok"])["odds"].mean().max()   # MaxAvgOdds                    else:                        val = d["odds"].mean()                                   # AvgOdds                    records.append(dict(model=model, stream=stream, attractors=att,                                        seed=seed, method=m, odds=val,                                        n_rows=len(d), err_frac=err_frac))interp = pd.DataFrame(records)print(f"{len(interp)} (model, stream, att, seed, method) points\n")print("dropped-cell fraction by method:")print(interp.groupby("method")["err_frac"].agg(["mean", "max"]).round(3))print("\nseeds per cell (should be 5):")print(interp.groupby(["model", "stream", "method", "attractors"]).size().unique())

In [ ]:
METHOD_COLORS = {"das": "tab:blue", "diffmean": "tab:orange",                 "probe": "tab:green", "patch_k128": "tab:purple"}PLOT_METHODS = SITE_METHODS + CIRCUIT_METHODSfig, axes = plt.subplots(2, 2, figsize=(12, 7.5), sharex=True, sharey=True)rng = np.random.default_rng(0)for ax, (model, stream) in zip(axes.ravel(),                               [(m, s) for m in MODELS for s in STREAMS]):    width = 0.8 / len(PLOT_METHODS)    for j, m in enumerate(PLOT_METHODS):        means, cis, xs = [], [], []        for att in range(4):            vals = interp.loc[(interp["model"] == model) & (interp["stream"] == stream)                              & (interp["method"] == m) & (interp["attractors"] == att),                              "odds"]            mu, ci = mean_ci(vals)            means.append(mu); cis.append(ci)            xpos = att + (j - (len(PLOT_METHODS) - 1) / 2) * width            xs.append(xpos)            ax.scatter(xpos + rng.uniform(-0.02, 0.02, len(vals)), vals,                       s=7, color="k", alpha=0.30, zorder=3, linewidths=0)        ax.bar(xs, means, width * 0.9, yerr=cis, capsize=3, zorder=2,               color=METHOD_COLORS[m], alpha=0.85,               label=m if (model, stream) == (MODELS[0], STREAMS[0]) else None)    ax.axhline(0, color="k", lw=0.8)    ax.set_title(f"{model} - {stream}", fontsize=11)    ax.set_xticks(range(4))    ax.grid(axis="y", alpha=0.25)for ax in axes[1]:    ax.set_xlabel("attractors")for ax in axes[:, 0]:    ax.set_ylabel("odds (log odds-ratio)")fig.legend(loc="upper center", ncol=len(PLOT_METHODS), frameon=False,           fontsize=9, bbox_to_anchor=(0.5, 1.005))fig.suptitle(f"Interp seed replicates, {SPLIT} split: MaxAvgOdds (das/diffmean/probe) and "             f"AvgOdds (patch_k128)\nmean +/- 95% CI over 5 seeds; dots = individual seeds",             fontsize=11, y=1.06)fig.tight_layout()out_pdf = os.path.join(FIG_DIR, "interp_seed_odds_ci.pdf")fig.savefig(out_pdf, bbox_inches="tight")print("wrote", out_pdf)plt.show()

In [ ]:
# Numeric companion to the figure: mean [lo, hi] per cell.tab = (interp.groupby(["model", "stream", "method", "attractors"])["odds"]             .agg(mean="mean", sd=lambda s: s.std(ddof=1), n="count"))tab["ci95"] = 1.96 * tab["sd"] / np.sqrt(tab["n"])tab["lo"] = tab["mean"] - tab["ci95"]tab["hi"] = tab["mean"] + tab["ci95"]print(tab[["mean", "lo", "hi", "n"]].round(3).to_string())